In [ ]:
%pip install -q "fastapi==0.135.1" "uvicorn==0.42.0" "websockets==15.0.1"


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()

_key = (_secrets.get_secret("RENDEZVOUS_KEY") or "").strip().strip("\"'")
assert _key, "RENDEZVOUS_KEY secret is empty - attach it under Add-ons > Secrets"
os.environ["RENDEZVOUS_KEY"] = _key
print(f"RENDEZVOUS_KEY loaded: length={len(_key)}, preview={_key[:4]}...{_key[-4:]}")

# Optional secret: point at a different rendezvous server. The default is the
# tailscale-funnel address of the server this notebook was built against.
try:
    _url = (_secrets.get_secret("RENDEZVOUS_URL") or "").strip().strip("\"'")
except Exception:
    _url = ""  # secret label not attached — fall back to the default
os.environ["RENDEZVOUS_URL"] = _url or os.environ.get(
    "RENDEZVOUS_URL", "wss://openclaw.tail2e2684.ts.net/connect"
)
print("RENDEZVOUS_URL:", os.environ["RENDEZVOUS_URL"])


In [ ]:
"""
kaggle_proxy.py — OpenAI-compatible FastAPI proxy for Kaggle Benchmarks models.

Run the notebook's installation and secret-setup cells first, then this cell.
When pasted into a cell, these functions are already defined: do not import
kaggle_proxy again.

Start locally with server = serve(); client = openai_client().
Expose it with bridge = connect(); remote clients then call the rendezvous
server's public URL, NOT this notebook directly.

Remote clients authenticate to the rendezvous server with their own client
key. PROXY_API_KEY only gates the loopback server inside this notebook; the
bridge injects it into relayed requests, so it never leaves the session.
PROXY_API_KEY is generated in memory unless provided through the environment.
Copy it privately if needed; do not print it into saved notebook outputs.

Tool schemas, tool-result messages, and image content parts pass through.
Support depends on the selected upstream model. Tools are not auto-executed.
Remote image URLs may need to be converted to inline base64 data URLs.
MODEL_ALIASES maps effective IDs to canonical Kaggle Model Proxy IDs.
The initial mappings come from Kaggle's model catalog; edit this dictionary
to add or override aliases. Configured notebook models also get short aliases.
The /v1/models list exposes effective IDs with their upstream_model mapping.
Full provider/model@version IDs remain accepted unchanged.
Model gate: set MODEL_ALLOWLIST (only these run) OR MODEL_BLOCKLIST
(everything but these runs) in the environment, comma-separated
ids/aliases. Both unset allows everything; allowlist wins if both set.

Stop with disconnect(bridge), then stop_server(server).
The notebook's interactive session must remain alive to keep serving.
"""

import hmac
import os
import secrets
import threading
import time
from typing import Any

import kaggle_benchmarks as kbench
import openai
from fastapi import APIRouter, Depends, FastAPI, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, ConfigDict


def proxy_api_key():
    return os.environ.setdefault("PROXY_API_KEY", secrets.token_urlsafe(32)).strip()


PROXY_API_KEY = proxy_api_key()
MODEL_ALIASES = {
    "gemini-3.8-flash": "google/gemini-3.8-flash",
    "claude-haiku-4-5": "anthropic/claude-haiku-4-5@20251001",
    "claude-haiku-5-5": "anthropic/claude-haiku-5-5",
    "claude-opus-4-5": "anthropic/claude-opus-4-5@20251101",
    "claude-opus-4-6": "anthropic/claude-opus-4-6@default",
    "claude-opus-4-7": "anthropic/claude-opus-4-7@default",
    "claude-opus-4-8": "anthropic/claude-opus-4-8@default",
    "claude-opus-5": "anthropic/claude-opus-5@default",
    "claude-opus-5-5": "anthropic/claude-opus-5-5@default",
    "claude-sonnet-4-6": "anthropic/claude-sonnet-4-6@default",
    "claude-sonnet-5": "anthropic/claude-sonnet-5@default",
    "claude-sonnet-5-5": "anthropic/claude-sonnet-5-5@default",
    "gemini-2.5-flash": "google/gemini-2.5-flash",
    "gemini-2.5-pro": "google/gemini-2.5-pro",
    "gemini-3-flash-preview": "google/gemini-3-flash-preview",
    "gemini-3.1-flash-lite-preview": "google/gemini-3.1-flash-lite-preview",
    "gemini-3.1-pro-preview": "google/gemini-3.1-pro-preview",
    "gemini-3.5-flash-lite": "google/gemini-3.5-flash-lite",
    "gemini-3.6-flash": "google/gemini-3.6-flash",
    "gemini-4-argon": "google/gemini-4-argon",
    "gemma-4-26b-a4b": "google/gemma-4-26b-a4b",
    "gemma-4-31b": "google/gemma-4-31b",
    "gpt-5.4-2026-03-05": "openai/gpt-5.4-2026-03-05",
    "gpt-5.4-mini-2026-03-17": "openai/gpt-5.4-mini-2026-03-17",
    "gpt-5.4-nano-2026-03-17": "openai/gpt-5.4-nano-2026-03-17",
    "gpt-5.5-2026-04-23": "openai/gpt-5.5-2026-04-23",
    "gpt-5.6-luna": "openai/gpt-5.6-luna",
    "gpt-5.6-sol": "openai/gpt-5.6-sol",
    "gpt-5.6-terra": "openai/gpt-5.6-terra",
    "gpt-6-astra": "openai/gpt-6-astra",
    "gpt-6-luna": "openai/gpt-6-luna",
    "gpt-6-sol": "openai/gpt-6-sol",
    "gpt-6.1-sol": "openai/gpt-6.1-sol",
    "gpt-oss-120b": "openai/gpt-oss-120b",
    "grok-4.5-0708": "xai/grok-4.5-0708",
    "grok-4.6": "xai/grok-4.6",
}

# ---------------------------------------------------------------------------
# Model Proxy access — every kbench LLMChat wraps the same configured
# openai.OpenAI client (base_url = {MODEL_PROXY_URL}/openapi), so one client
# can serve any model slug the proxy supports.
# ---------------------------------------------------------------------------

def _shared_client():
    for llm in [kbench.llm, *getattr(kbench, "llms", {}).values()]:
        client = getattr(llm, "client", None)
        if client is not None and hasattr(client, "chat"):
            return client
    raise HTTPException(
        503,
        "No OpenAI-compatible Model Proxy client found — set MODEL_PROXY_URL "
        "and MODEL_PROXY_API_KEY, or run inside a Kaggle Benchmarks notebook.",
    )


def _default_model():
    slug = getattr(kbench.llm, "model", None)
    if not slug:
        raise HTTPException(400, "Request must include `model` (no default set).")
    return slug


def _model_aliases():
    slugs = set(getattr(kbench, "llms", None) or {})
    slugs.update(MODEL_ALIASES.values())
    default = getattr(kbench.llm, "model", None)
    if default:
        slugs.add(default)
    aliases = {}
    for slug in sorted(slugs):
        alias = slug.rsplit("/", 1)[-1].removesuffix("@default")
        if alias in aliases and aliases[alias] != slug:
            aliases[alias] = None
        else:
            aliases[alias] = slug
    aliases.update(MODEL_ALIASES)
    return aliases


def _resolve_model(model):
    if "/" in model:
        return model
    aliases = _model_aliases()
    if model in aliases and aliases[model] is None:
        raise HTTPException(
            400, f"Ambiguous model alias {model!r}; use a full upstream model ID"
        )
    return aliases.get(model, model)


def _model_allowed(model):
    """Model gate — set MODEL_ALLOWLIST (only these run) OR MODEL_BLOCKLIST
    (everything but these runs), comma-separated ids/aliases. If both are
    set the allowlist wins. Both empty = allow everything."""
    allow = {
        m.strip() for m in os.environ.get("MODEL_ALLOWLIST", "").split(",")
        if m.strip()
    }
    block = {
        m.strip() for m in os.environ.get("MODEL_BLOCKLIST", "").split(",")
        if m.strip()
    }
    if not allow and not block:
        return True
    aliases = _model_aliases()
    candidates = {model, aliases.get(model)}
    candidates.update(a for a, u in aliases.items() if u == model)
    candidates.discard(None)
    if allow:
        return bool(candidates & allow)
    return not bool(candidates & block)


# ---------------------------------------------------------------------------
# FastAPI app — OpenAI wire format, verbatim passthrough
# ---------------------------------------------------------------------------

def require_api_key(
    credentials: HTTPAuthorizationCredentials | None = Depends(
        HTTPBearer(auto_error=False)
    ),
):
    if credentials is None or not hmac.compare_digest(
        credentials.credentials.encode(), proxy_api_key().encode()
    ):
        raise HTTPException(
            401, "Invalid proxy API key", headers={"WWW-Authenticate": "Bearer"}
        )


router = APIRouter(dependencies=[Depends(require_api_key)])


class ChatCompletionRequest(BaseModel):
    """OpenAI chat.completions.create params. Unknown fields pass through."""

    model_config = ConfigDict(extra="allow")

    model: str | None = None
    messages: list[dict[str, Any]]
    stream: bool = False
    temperature: float | None = None
    top_p: float | None = None
    max_tokens: int | None = None
    max_completion_tokens: int | None = None
    n: int | None = None
    seed: int | None = None
    stop: str | list[str] | None = None
    presence_penalty: float | None = None
    frequency_penalty: float | None = None
    logit_bias: dict[str, float] | None = None
    logprobs: bool | None = None
    top_logprobs: int | None = None
    user: str | None = None
    tools: list[dict[str, Any]] | None = None
    tool_choice: str | dict[str, Any] | None = None
    response_format: dict[str, Any] | None = None
    reasoning_effort: str | None = None
    stream_options: dict[str, Any] | None = None
    metadata: dict[str, str] | None = None


@router.get("/models")
def list_models():
    return {
        "object": "list",
        "data": [
            {
                "id": alias, "object": "model", "created": 0,
                "owned_by": "kaggle", "upstream_model": upstream,
            }
            for alias, upstream in sorted(_model_aliases().items())
            if upstream is not None
        ],
    }


def _sse(upstream):
    """Yield an openai.Stream as OpenAI-style SSE chunks."""
    try:
        for chunk in upstream:
            yield f"data: {chunk.model_dump_json(exclude_none=True)}\n\n"
    finally:
        close = getattr(upstream, "close", None)
        if callable(close):
            close()
    yield "data: [DONE]\n\n"


@router.post("/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    payload = req.model_dump(exclude_unset=True)
    stream = payload.pop("stream", False)
    model = _resolve_model(payload.pop("model", None) or _default_model())
    if not _model_allowed(model):
        raise HTTPException(403, f"Model {model!r} is blocked by the "
                                 "model allow/block list")

    try:
        upstream = _shared_client().chat.completions.create(
            model=model, stream=stream, **payload
        )
    except openai.APIStatusError as exc:
        if exc.status_code in (401, 403):
            return JSONResponse(
                status_code=502,
                content={"error": {"message": (
                    f"UPSTREAM Kaggle Model Proxy rejected its credentials "
                    f"({exc.status_code}); your PROXY_API_KEY was fine. Check "
                    f"MODEL_PROXY_API_KEY / that this is a Benchmarks notebook.")}},
            )
        body = exc.body
        if not isinstance(body, dict):
            body = {"message": "Upstream model request failed"}
        if "error" not in body:
            body = {"error": body}
        return JSONResponse(status_code=exc.status_code, content=body)
    except openai.APIConnectionError:
        return JSONResponse(
            status_code=502,
            content={"error": {"message": "Could not reach Kaggle Model Proxy"}},
        )

    if stream:
        return StreamingResponse(_sse(upstream), media_type="text/event-stream")

    # Real ChatCompletion, serialized faithfully (nulls included, like OpenAI).
    return upstream.model_dump(mode="json")


app = FastAPI(title="Kaggle Benchmarks Model Proxy", version="1.0.0")
app.include_router(router, prefix="/v1")


# ---------------------------------------------------------------------------
# Server + client helpers
# ---------------------------------------------------------------------------

def serve(host="127.0.0.1", port=8000, **uvicorn_kwargs):
    """
    Run the proxy in a daemon thread (notebook-friendly — the cell returns
    immediately). Returns the uvicorn.Server; base URL is
    http://{host}:{port}/v1.
    """
    import asyncio

    import uvicorn

    options = {"log_level": "warning", "loop": "asyncio", **uvicorn_kwargs}
    config = uvicorn.Config(app, host=host, port=port, **options)
    server = uvicorn.Server(config)
    errors = []

    def _run():
        # ipykernel patches asyncio.run without loop_factory support, which
        # breaks uvicorn.Server.run() — drive serve() on our own loop instead.
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(server.serve())
        except BaseException as exc:
            errors.append(exc)
        finally:
            loop.run_until_complete(loop.shutdown_asyncgens())
            asyncio.set_event_loop(None)
            loop.close()

    server.thread = threading.Thread(
        target=_run, daemon=True, name="kbench-openai-proxy"
    )
    server.thread.start()

    deadline = time.monotonic() + 30
    while time.monotonic() < deadline:
        if errors:
            raise RuntimeError("Proxy startup failed") from errors[0]
        if not server.thread.is_alive():
            raise RuntimeError("Proxy thread exited during startup")
        if server.started:
            return server
        time.sleep(0.05)
    server.should_exit = True
    raise TimeoutError("Proxy did not start within 30 seconds")


def stop_server(server):
    server.should_exit = True
    server.thread.join(timeout=10)
    if server.thread.is_alive():
        raise RuntimeError("Proxy is still shutting down")


def openai_client(base_url="http://127.0.0.1:8000/v1", **kwargs):
    """openai.OpenAI pointed at the proxy (local base URL by default)."""
    import openai

    kwargs.setdefault("api_key", proxy_api_key())
    return openai.OpenAI(base_url=base_url, **kwargs)


class ChatSession:
    """
    Minimal stateful multi-turn helper over the proxy.

        chat = ChatSession(model="google/gemini-2.5-flash", system="You are terse.")
        chat.ask("Hi!")        # -> str
        chat.ask("And you?")   # history carried automatically
        chat.messages          # full OpenAI-style history (tool_calls preserved)
        chat.reset()           # clear; chat.reset(system="...") to re-seed
    """

    def __init__(self, model=None, system=None,
                 base_url="http://127.0.0.1:8000/v1", **client_kwargs):
        import os

        if model is None:
            model = (
                getattr(kbench.llm, "model", None)
                or os.environ.get("LLM_DEFAULT")
            )
        if not model:
            raise ValueError("ChatSession requires a model slug.")

        self._oai = openai_client(base_url=base_url, **client_kwargs)
        self.model = model
        self.messages = [{"role": "system", "content": system}] if system else []

    @property
    def history(self):
        return list(self.messages)

    def ask(self, content, **kwargs):
        """Append a user message, get a completion, append the reply, return text."""
        if kwargs.get("stream"):
            raise ValueError("Use create(stream=True) for streaming")
        messages = [*self.messages, {"role": "user", "content": content}]
        resp = self._oai.chat.completions.create(
            model=self.model, messages=messages, **kwargs
        )
        msg = resp.choices[0].message
        # model_dump preserves tool_calls etc. for faithful multi-turn history.
        self.messages = [*messages, msg.model_dump(exclude_none=True)]
        return msg.content or ""

    def send(self, content, **kwargs):
        """Alias for ask()."""
        return self.ask(content, **kwargs)

    def create(self, **kwargs):
        """Raw passthrough on current history WITHOUT appending the exchange."""
        return self._oai.chat.completions.create(
            model=self.model, messages=list(self.messages), **kwargs
        )

    def reset(self, system=None):
        self.messages = [{"role": "system", "content": system}] if system else []
        return self


In [ ]:
"""Start locally. On reruns, stop this notebook's previous server first."""

if "server" in globals() and getattr(server, "thread", None):
    stop_server(server)
server = serve(port=8000)
with openai_client() as local_client:
    model_ids = [model.id for model in local_client.models.list().data]
print("Local proxy is ready. Configured models:", model_ids)


In [ ]:
"""
kaggle_bridge.py — outbound WebSocket bridge to the rendezvous server.

Run the proxy cell first (serve() must be running). Then:

    bridge = connect(port=8000)          # dials out, relays HTTP over WS
    bridge.public_base_url               # URL remote clients should use
    disconnect(bridge)                   # tear down

Configuration (Kaggle secrets -> env vars):
    RENDEZVOUS_KEY   required bearer key the server authorizes with
    RENDEZVOUS_URL   ws(s):// URL of the server's /connect endpoint
Both may also be passed explicitly as connect(url=..., key=...).

The bridge authenticates to the rendezvous server with the rendezvous key,
then forwards each relayed HTTP request to the local proxy on
http://127.0.0.1:{port} — injecting PROXY_API_KEY itself, so remote clients
never see it.

Failsafes:
  * Session drain — Kaggle sessions die at ~12h; the bridge drains at
    T-10min (env BRIDGE_DRAIN_AFTER_S, default 42600s): it tells the server
    to stop routing, returns 503 for new requests, lets in-flight work
    finish, then disconnects. connect(drain_after=S) to override/0 to off.
  * Reconnect backoff with jitter (thundering-herd safe).
  * Concurrency cap — at most max_concurrent in-flight relays (default 8);
    excess requests get a 429 after queue_timeout instead of piling up.
  * Loopback timeouts — dead local proxy fails fast (connect=5s, pool=5s,
    write=10s); streaming reads stay open (read=None).
The notebook session must stay alive while bridging.
"""

import asyncio
import base64
import json
import os
import random
import threading
import time

_HOP_BY_HOP = {
    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailer", "transfer-encoding", "upgrade",
    "content-length", "content-encoding", "host", "authorization",
}


class Bridge:
    """Handle for a running bridge; returned by connect()."""

    def __init__(self, url, key, port, api_key, name, reconnect,
                 drain_after, max_concurrent, queue_timeout):
        self.url = url
        self.key = key
        self.port = port
        self.api_key = api_key
        self.name = name
        self.reconnect = reconnect
        self.max_concurrent = max_concurrent
        self.queue_timeout = queue_timeout
        self._drain_at = (
            time.monotonic() + drain_after if drain_after else None
        )
        self.draining = False
        self.connected = threading.Event()
        self.notebook_id = None
        self.public_base_url = None
        self.error = None
        self.relayed = 0
        self._should_run = True
        self._tasks = {}
        self._ws = None
        self._loop = None
        self._send_lock = None
        self._sema = None
        self._client = None
        self._drain_task = None
        self.thread = threading.Thread(
            target=self._thread_main, daemon=True, name="kbench-ws-bridge"
        )

    # ------------------------------------------------------------------
    # thread plumbing — same pattern as serve(): own loop in a daemon thread
    # ------------------------------------------------------------------

    def start(self):
        self.thread.start()
        return self

    def _thread_main(self):
        self._loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self._loop)
        try:
            self._loop.run_until_complete(self._run())
        finally:
            self._loop.run_until_complete(self._loop.shutdown_asyncgens())
            asyncio.set_event_loop(None)
            self._loop.close()

    # ------------------------------------------------------------------
    # main loop: connect -> session -> (reconnect with jitter)
    # ------------------------------------------------------------------

    @staticmethod
    def _retry_delay(backoff):
        return min(backoff * random.uniform(0.5, 1.5), 30.0)

    async def _run(self):
        import websockets

        backoff = 1.0
        while self._should_run:
            headers = {
                "Authorization": f"Bearer {self.key}",
                "X-Notebook-Name": self.name,
            }
            connect_kwargs = dict(
                max_size=64 * 1024 * 1024,
                ping_interval=20,
                ping_timeout=30,
                open_timeout=15,
                close_timeout=5,
            )
            try:
                try:
                    ws = await websockets.connect(
                        self.url, additional_headers=headers, **connect_kwargs
                    )
                except TypeError:  # websockets <14: extra_headers
                    ws = await websockets.connect(
                        self.url, extra_headers=headers, **connect_kwargs
                    )
            except Exception as exc:
                status = getattr(exc, "status_code", None) or getattr(
                    getattr(exc, "response", None), "status_code", None
                )
                self.error = f"connect failed: {exc!r}"
                if status in (401, 403):
                    self.error = (
                        f"rendezvous rejected the key (HTTP {status}) — "
                        "check RENDEZVOUS_KEY"
                    )
                    print(f"[bridge] {self.error}")
                    self._should_run = False
                    self.connected.set()  # unblock ready() callers
                    return
                if not (self.reconnect and self._should_run):
                    self.connected.set()
                    return
                delay = self._retry_delay(backoff)
                print(f"[bridge] {self.error}; retrying in {delay:.0f}s")
                await asyncio.sleep(delay)
                backoff = min(backoff * 2, 30)
                continue

            try:
                import httpx
                self._ws = ws
                self._send_lock = asyncio.Lock()
                self._sema = asyncio.Semaphore(self.max_concurrent)
                self._client = httpx.AsyncClient(
                    # dead loopback fails fast; streaming reads stay open
                    timeout=httpx.Timeout(connect=5, read=None, write=10,
                                          pool=5),
                    limits=httpx.Limits(max_connections=self.max_concurrent),
                )
                self._drain_task = asyncio.ensure_future(self._drain_timer(ws))
                await self._session(ws)
                backoff = 1.0
            except Exception as exc:
                self.error = f"session ended: {exc!r}"
            finally:
                self._ws = None
                self.connected.clear()
                if self._drain_task:
                    self._drain_task.cancel()
                    self._drain_task = None
                for task in self._tasks.values():
                    task.cancel()
                self._tasks.clear()
                if self._client:
                    try:
                        await self._client.aclose()
                    except Exception:
                        pass
                    self._client = None
                try:
                    await ws.close()
                except Exception:
                    pass
            if not (self.reconnect and self._should_run) or self.draining:
                break
            delay = self._retry_delay(backoff)
            print(f"[bridge] disconnected; retrying in {delay:.0f}s")
            await asyncio.sleep(delay)
            backoff = min(backoff * 2, 30)

    async def _drain_timer(self, ws):
        """At the session-limit deadline: tell the server to stop routing,
        refuse new work, let in-flight requests finish, then disconnect."""
        if self._drain_at is None:
            return
        delay = self._drain_at - time.monotonic()
        if delay > 0:
            await asyncio.sleep(delay)
        self.draining = True
        print("[bridge] draining — refusing new requests (session limit)")
        try:
            await self._send(ws, {"type": "drain"})
        except Exception:
            pass
        while self._tasks:  # don't kill mid-run work
            await asyncio.sleep(2)
        self._should_run = False
        try:
            await ws.close()
        except Exception:
            pass

    async def _session(self, ws):
        async for raw in ws:
            try:
                msg = json.loads(raw)
            except Exception:
                continue
            mtype = msg.get("type")
            if mtype == "welcome":
                self.notebook_id = msg.get("notebook_id")
                self.public_base_url = msg.get("public_base_url")
                self.error = None
                self.connected.set()
                print(
                    f"[bridge] connected as {self.notebook_id}"
                    + (f" -> {self.public_base_url}"
                       if self.public_base_url else "")
                )
            elif mtype == "http_request":
                if self.draining:
                    await self._send(ws, {
                        "type": "http_error", "id": msg["id"], "status": 503,
                        "message": "notebook is draining (session limit)",
                    })
                else:
                    task = asyncio.ensure_future(self._handle(ws, msg))
                    self._tasks[msg["id"]] = task
            elif mtype == "http_cancel":
                task = self._tasks.pop(msg.get("id"), None)
                if task:
                    task.cancel()
            elif mtype == "drain":  # server-initiated drain
                self.draining = True
            elif mtype == "bye":
                return

    # ------------------------------------------------------------------
    # request relay: WS message -> local HTTP -> WS messages
    # ------------------------------------------------------------------

    async def _send(self, ws, msg):
        async with self._send_lock:
            await ws.send(json.dumps(msg))

    def _local_headers(self, headers):
        out = {
            k: v for k, v in (headers or {}).items()
            if k.lower() not in _HOP_BY_HOP
        }
        if self.api_key:
            out["Authorization"] = f"Bearer {self.api_key}"
        return out

    async def _handle(self, ws, msg):
        rid = msg["id"]
        try:
            await asyncio.wait_for(self._sema.acquire(), self.queue_timeout)
        except Exception:
            try:
                await self._send(ws, {
                    "type": "http_error", "id": rid, "status": 429,
                    "message": "notebook concurrency limit reached",
                })
            finally:
                self._tasks.pop(rid, None)
            return
        self.relayed += 1
        try:
            body = base64.b64decode(msg.get("body_b64") or b"")
            async with self._client.stream(
                msg.get("method", "GET"),
                f"http://127.0.0.1:{self.port}{msg.get('path', '/')}",
                headers=self._local_headers(msg.get("headers")),
                content=body or None,
            ) as resp:
                headers = {
                    k: v for k, v in resp.headers.items()
                    if k.lower() not in _HOP_BY_HOP
                }
                await self._send(ws, {
                    "type": "http_response_start",
                    "id": rid, "status": resp.status_code,
                    "headers": headers,
                })
                async for chunk in resp.aiter_bytes(65536):
                    await self._send(ws, {
                        "type": "http_response_chunk",
                        "id": rid,
                        "data_b64": base64.b64encode(chunk).decode(),
                    })
            await self._send(ws, {"type": "http_response_end", "id": rid})
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            try:
                await self._send(ws, {
                    "type": "http_error", "id": rid,
                    "status": 502, "message": f"bridge relay failed: {exc!r}",
                })
            except Exception:
                pass
        finally:
            self._sema.release()
            self._tasks.pop(rid, None)

    # ------------------------------------------------------------------
    # public API
    # ------------------------------------------------------------------

    def ready(self, timeout=30):
        """Block until the first successful connect; raises on failure."""
        if not self.connected.wait(timeout):
            raise TimeoutError(
                f"bridge did not connect within {timeout}s: {self.error}"
            )
        if self.error and self.notebook_id is None:
            raise RuntimeError(self.error)
        return self

    def close(self):
        """Stop reconnecting and drop the current connection."""
        self._should_run = False

        async def _shutdown():
            if self._drain_task:
                self._drain_task.cancel()
            for task in self._tasks.values():
                task.cancel()
            if self._ws is not None:
                try:
                    await self._ws.close()
                except Exception:
                    pass

        loop = self._loop
        if loop and loop.is_running():
            fut = asyncio.run_coroutine_threadsafe(_shutdown(), loop)
            try:
                fut.result(timeout=10)
            except Exception:
                pass
        self.thread.join(timeout=10)

    def __repr__(self):
        state = "draining" if self.draining else (
            "connected" if self.connected.is_set() else "disconnected"
        )
        return f"<Bridge {self.notebook_id or '-'} {state} url={self.url!r}>"


def connect(url=None, key=None, port=8000, api_key=None, name=None,
            reconnect=True, drain_after=None, max_concurrent=8,
            queue_timeout=30):
    """
    Start the rendezvous bridge in a daemon thread. Returns a Bridge.

    url/key default to the RENDEZVOUS_URL / RENDEZVOUS_KEY env vars (loaded
    from Kaggle secrets). api_key defaults to proxy_api_key() and is sent to
    the LOCAL proxy only — remote clients authenticate to the server with
    their own key.

    drain_after: seconds until the bridge starts draining new requests
    (default BRIDGE_DRAIN_AFTER_S env, else 42600 = 12h session - 10min;
    0 disables). max_concurrent bounds in-flight relays; queue_timeout is
    how long an over-capacity request waits before getting a 429.
    """
    url = url or os.environ.get("RENDEZVOUS_URL")
    key = key or os.environ.get("RENDEZVOUS_KEY")
    if api_key is None:
        api_key = proxy_api_key() if "proxy_api_key" in globals() else None
    if drain_after is None:
        drain_after = float(os.environ.get("BRIDGE_DRAIN_AFTER_S", "42600"))
    if not url:
        raise ValueError("Set RENDEZVOUS_URL (env or secret) or pass url=")
    if not key:
        raise ValueError("Set RENDEZVOUS_KEY (env or secret) or pass key=")
    if url.startswith("https://"):
        url = "wss://" + url[len("https://"):]
    elif url.startswith("http://"):
        url = "ws://" + url[len("http://"):]
    bridge = Bridge(url, key, port, api_key, name or "kaggle-notebook",
                    reconnect, drain_after or None, max_concurrent,
                    queue_timeout)
    return bridge.start()


def disconnect(bridge):
    """Stop the bridge returned by connect()."""
    if bridge is not None:
        bridge.close()


In [ ]:
"""Dial out to the rendezvous server.

Remote clients send OpenAI-style requests to the rendezvous server's public
URL (printed below) with a CLIENT_API_KEY as their Bearer token — not the
rendezvous key and not PROXY_API_KEY, both of which stay private.
Keep this interactive notebook session alive while serving.
"""

import os

if "bridge" in globals() and bridge is not None:
    disconnect(bridge)
bridge = connect(port=8000).ready(timeout=30)
print("Connected as:", bridge.notebook_id)
print("Public base URL for remote clients:",
      bridge.public_base_url
      or os.environ["RENDEZVOUS_URL"].replace("wss://", "https://")
         .replace("ws://", "http://").rstrip("/").removesuffix("/connect") + "/v1")


In [ ]:
"""Optional inference test: call smoke_test_model() to spend model quota.
Uses the loopback proxy; to test end-to-end through the rendezvous server,
call it from a remote client against the public base URL instead."""

def smoke_test_model(model=None):
    with openai_client() as client:
        response = client.chat.completions.create(
            model=model or _default_model(),
            messages=[{"role": "user", "content": "Reply with just: OK"}],
        )
    return response.choices[0].message.content


In [ ]:
"""Call shutdown_proxy() when finished."""

def shutdown_proxy():
    try:
        if "bridge" in globals() and bridge is not None:
            disconnect(bridge)
    finally:
        if "server" in globals():
            stop_server(server)
